## Libraries

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import STL
from sklearn.preprocessing import StandardScaler
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import Ridge
from scipy.stats import norm
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.neighbors import NearestNeighbors

## Config

In [ ]:
TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

TRAIN_END_DATE = pd.Timestamp("2022-03-31")
TEST_START_DATE = pd.Timestamp("2022-04-01")

# ---- INSERT YOUR BEST LAG SET & PARAMS HERE ----
best_lag_set = [1, 2, 3, 12, 24]
best_svm_params = {
    "gamma": 0.1,
    "n_components": 800,
    "C": 10.0,
    "epsilon": 0.05
}

# map SVM C to ridge alpha (roughly alpha ~ 1/C)
best_gamma       = best_svm_params["gamma"]
best_n_components = best_svm_params["n_components"]
best_alpha       = 1.0 / best_svm_params["C"]

# ---- FEATURES ----
continuous_cols = [
    "AverageNeighbourPrice","local_I","area_km2","centroid_x","centroid_y",
    "CoL_distance_km","LA_FE","sdlt_perc_threshold","dwelling_stock",
    "population","ashe_weekly","base_rate","claimant_count_prop",
    "planning_decisions_per_1000","planning_granted_prop",
    "rail_station_entry_exit","GDP","CPIH"
]

categorical_cols = [
    "LMIQuadrant__2","LMIQuadrant__3","LMIQuadrant__4",
    "Region_East of England","Region_London","Region_North East",
    "Region_North West","Region_South East","Region_South West",
    "Region_West Midlands","Region_Yorkshire and The Humber"
]

## Metric functions

In [ ]:
def mae(y,yhat): return np.mean(np.abs(y-yhat))
def rmse(y,yhat): return np.sqrt(np.mean((y-yhat)**2))
def smape(y,yhat,eps=1e-8):
    return 100*np.mean(2*np.abs(yhat-y)/(np.abs(y)+np.abs(yhat)+eps))

def mase(y,yhat,y_train,m=12,eps=1e-8):
    naive = np.abs(y_train[m:] - y_train[:-m])
    return np.mean(np.abs(y-yhat)) / (np.mean(naive)+eps)

def directional_accuracy(df,entity,time,y,yhat):
    def f(x):
        return np.mean(np.sign(x[y].diff()) == np.sign(x[yhat].diff()))
    return df.groupby(entity).apply(f).mean()

def growth_rate_error(df,entity,time,y,yhat,m=12):
    def f(x):
        return np.mean(np.abs((x[y].pct_change(m) - x[yhat].pct_change(m))))
    return df.groupby(entity).apply(f).mean()

def morans_i(residuals,xs,ys,k=5):
    N=len(residuals)
    X=residuals-np.mean(residuals)
    coords=np.column_stack([xs,ys])
    nbrs=NearestNeighbors(n_neighbors=k+1).fit(coords)
    _,idx=nbrs.kneighbors(coords)
    W=np.zeros((N,N))
    for i in range(N):
        W[i,idx[i][1:]]=1
    W=W/np.sum(W,axis=1,keepdims=True)
    num=np.sum(W*(X[:,None]*X[None,:]))
    den=np.sum(X**2)
    return (N/np.sum(W))*num/den

def crps_gaussian(y,mu,sigma,eps=1e-8):
    a=(y-mu)/(sigma+eps)
    return np.mean(sigma*(1/np.sqrt(np.pi)-2*norm.pdf(a)-a*(2*norm.cdf(a)-1)))

## Load data

In [ ]:
df = pd.read_excel("../../data/full_data.xlsx", parse_dates=[TIME_COL])
df = df.sort_values([ENTITY_COL,TIME_COL]).reset_index(drop=True)

df_train_all = df[df[TIME_COL] <= TRAIN_END_DATE].copy()
df_test      = df[df[TIME_COL] >= TEST_START_DATE].copy()

## Training

In [ ]:
df_train_all["stl_trend"]=np.nan
df_train_all["stl_seasonal"]=np.nan
df_train_all["stl_resid"]=np.nan

for la,sub in df_train_all.groupby(ENTITY_COL):
    sub=sub.sort_values(TIME_COL)
    if len(sub)<24: continue
    stl=STL(sub[TARGET_COL],period=12,robust=True).fit()
    df_train_all.loc[sub.index,"stl_trend"]=stl.trend
    df_train_all.loc[sub.index,"stl_seasonal"]=stl.seasonal
    df_train_all.loc[sub.index,"stl_resid"]=stl.resid

# Extend STL into test
df_test["stl_trend"]=np.nan
df_test["stl_seasonal"]=np.nan
df_test["stl_resid"]=0.0

for la in df_train_all[ENTITY_COL].unique():
    sub_train=df_train_all[df_train_all[ENTITY_COL]==la]
    sub_test =df_test[df_test[ENTITY_COL]==la]
    if sub_test.empty: continue
    season=sub_train["stl_seasonal"].dropna().values[-12:]
    reps=int(np.ceil(len(sub_test)/12))
    df_test.loc[sub_test.index,"stl_seasonal"]=np.tile(season,reps)[:len(sub_test)]
    t=sub_train["stl_trend"].dropna().values
    coef=np.polyfit(np.arange(len(t)),t,1)
    df_test.loc[sub_test.index,"stl_trend"]=coef[0]*np.arange(len(t),len(t)+len(sub_test))+coef[1]

# Combine & create lags
combined=pd.concat([df_train_all,df_test]).sort_values([ENTITY_COL,TIME_COL])
for lag in best_lag_set:
    for comp in ["stl_trend","stl_seasonal","stl_resid"]:
        combined[f"{comp}_lag{lag}"] = combined.groupby(ENTITY_COL)[comp].shift(lag)

lag_cols=[f"{c}_lag{l}" for c in ["stl_trend","stl_seasonal","stl_resid"] for l in best_lag_set]
feature_cols=continuous_cols+categorical_cols+lag_cols

df_train=combined[combined[TIME_COL]<=TRAIN_END_DATE].dropna(subset=feature_cols)
df_test =combined[combined[TIME_COL]>=TEST_START_DATE].dropna(subset=feature_cols)

# =========================================================
# SCALE X AND y
# =========================================================
X_train=df_train[feature_cols]
X_test =df_test[feature_cols]

scaler=StandardScaler()
X_train[continuous_cols+lag_cols]=scaler.fit_transform(X_train[continuous_cols+lag_cols])
X_test[continuous_cols+lag_cols] =scaler.transform(X_test[continuous_cols+lag_cols])

y_train_raw=df_train[TARGET_COL].values.reshape(-1,1)
y_test_true=df_test[TARGET_COL].values

y_scaler=StandardScaler()
y_train_scaled=y_scaler.fit_transform(y_train_raw).ravel()

## Final fit

In [ ]:
rff = RBFSampler(gamma=best_gamma,
                 n_components=best_n_components,
                 random_state=42)

Z_train = rff.fit_transform(X_train)
Z_test  = rff.transform(X_test)

# Ridge with alpha ~ 1/C from SVM
ridge = Ridge(alpha=best_alpha)
ridge.fit(Z_train, y_train_scaled)

y_pred_scaled = ridge.predict(Z_test)
y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1,1)).ravel()

df_test["y_pred"]=y_pred
df_test["resid"]=y_test_true-y_pred

## Evaluation

In [ ]:
# =========================================================
# UNCERTAINTY (PICP, PIW, CRPS)
# =========================================================
Sigma_w = np.linalg.inv(Z_train.T @ Z_train + best_alpha*np.eye(Z_train.shape[1]))
train_resid_scaled = y_train_scaled - ridge.predict(Z_train)
sigma2_hat = np.var(train_resid_scaled)

pred_var_scaled = np.sum((Z_test @ Sigma_w)*Z_test, axis=1)*sigma2_hat + sigma2_hat
y_std = np.sqrt(pred_var_scaled) * y_scaler.scale_[0]

z = 1.96
y_lower = y_pred - z*y_std
y_upper = y_pred + z*y_std

PICP = np.mean((y_test_true >= y_lower) & (y_test_true <= y_upper))
PIW  = np.mean(y_upper - y_lower)
CRPS = crps_gaussian(y_test_true, y_pred, y_std)

# =========================================================
# GLOBAL METRICS
# =========================================================
global_mae   = mae(y_test_true,y_pred)
global_rmse  = rmse(y_test_true,y_pred)
global_smape = smape(y_test_true,y_pred)
global_mase  = mase(y_test_true,y_pred,y_train_raw.ravel())

# =========================================================
# ACROSS-LA CONSISTENCY
# =========================================================
la_mae=df_test.groupby(ENTITY_COL).apply(lambda x: mae(x[TARGET_COL],x["y_pred"]))
median_mae=np.median(la_mae)
p75_mae=np.percentile(la_mae,75)

# =========================================================
# SPATIO-TEMPORAL DIAGNOSTICS
# =========================================================
la_resid = df_test.groupby(ENTITY_COL)["resid"].mean()

centroids = (
    df_test.drop_duplicates(ENTITY_COL)
           .set_index(ENTITY_COL)[["centroid_x","centroid_y"]]
           .loc[la_resid.index]   # <-- critical alignment
)

I_moran = morans_i(
    la_resid.values,
    centroids["centroid_x"].values,
    centroids["centroid_y"].values
)

monthly_resid=df_test.groupby(TIME_COL)["resid"].mean()
lb_res=acorr_ljungbox(monthly_resid,lags=[12],return_df=True)
q_stat=float(lb_res["lb_stat"].iloc[0])
p_val =float(lb_res["lb_pvalue"].iloc[0])

# =========================================================
# DIRECTION & GROWTH
# =========================================================
dir_acc = directional_accuracy(df_test,ENTITY_COL,TIME_COL,TARGET_COL,"y_pred")
gre_mae = growth_rate_error(df_test,ENTITY_COL,TIME_COL,TARGET_COL,"y_pred")

# =========================================================
# SAVE RESULTS
# =========================================================
summary_df=pd.DataFrame([{
    "model":"SparseGP_RFF",
    "lag_set":str(best_lag_set),
    "params":str(best_svm_params),
    "MAE":global_mae,
    "RMSE":global_rmse,
    "sMAPE":global_smape,
    "MASE":global_mase,
    "Median_LA_MAE":median_mae,
    "P75_LA_MAE":p75_mae,
    "Morans_I":I_moran,
    "LjungBox_Q12":q_stat,
    "LjungBox_p":p_val,
    "Directional_Accuracy":dir_acc,
    "GrowthRateError_MAE":gre_mae,
    "PICP_95":PICP,
    "PIW_95":PIW,
    "CRPS":CRPS
}])

summary_df.to_excel("../../results/sparsegp_final_test_results.xlsx",index=False)
la_mae.reset_index().to_excel("../../results/sparsegp_la_mae.xlsx",index=False)

print("\n=== FINAL SPARSE GP TEST RESULTS SAVED ===")
print(summary_df.T)
